In [1]:
# Install required libraries quietly
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa tensorboard bitsandbytes

import datasets
import evaluate
import re
import gc
import os

# Disable Hugging Face caching to prevent disk space from filling up with temporary files
datasets.disable_caching()

# Dictionary to map numbers to their spoken Shona words
NUM_MAP = {
    "1": "motsi", "2": "piri", "3": "tatu", "4": "ina", "5": "shanu",
    "6": "tanhatu", "7": "nomwe", "8": "tsere", "9": "pfumbamwe", "0": "zero"
}

def clean_text_pipeline(text):
    # Handle empty text
    if text is None: return ""
    text = text.lower()

    # Replace numbers with their word equivalents
    for num, word in NUM_MAP.items():
        text = text.replace(num, f" {word} ")

    # Strip all punctuation using regex (keeps only words and spaces)
    text = re.sub(r'[^​\w\s]', '', text)

    # Remove extra white spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def create_dataset_ultra_safe(dataset_id, language, sample_rate, max_audio_len=30.0, min_audio_len=5.0):
    print(f"Loading {language} dataset...")

    # Explicitly specify data files to avoid downloading the massive unlabeled dataset
    train_pattern = f"data/ASR/{language}/{language}-train-*.parquet"
    val_pattern = f"data/ASR/{language}/{language}-validation-*.parquet"

    ds_train = datasets.load_dataset(dataset_id, name=f"{language}_asr", data_files={"train": train_pattern}, split="train")
    ds_val = datasets.load_dataset(dataset_id, name=f"{language}_asr", data_files={"validation": val_pattern}, split="validation")

    # Combine train and validation
    ds_combined = datasets.concatenate_datasets([ds_train, ds_val])

    print("Resampling audio...")
    ds_combined = ds_combined.cast_column("audio", datasets.Audio(sampling_rate=sample_rate))

    print("Filtering audio lengths...")
    def filter_audio_length(example):
        arr = example["audio"]["array"]
        duration = len(arr) / sample_rate
        return min_audio_len <= duration <= max_audio_len

    ds_filtered = ds_combined.filter(filter_audio_length)

    # Delete large variables to free up RAM immediately
    del ds_combined
    gc.collect()

    print("Cleaning text...")
    def clean_batch_text(batch):
        batch["transcription"] = [clean_text_pipeline(t) for t in batch["transcription"]]
        return batch

    ds_cleaned = ds_filtered.map(clean_batch_text, batched=True)

    # Free up RAM again
    del ds_filtered
    gc.collect()

    print("Splitting dataset 80/20...")
    final_splits = ds_cleaned.train_test_split(test_size=0.2, seed=42)
    return final_splits

lang_code = "sna"
sr = 16000
# Limit to 15 seconds because MMS-1B is a massive 1-billion parameter model
max_len = 15.0

# Check if we already processed the data to save time on restarts
if os.path.exists("./waxal_processed_dataset"):
    print("Loading saved dataset from disk...")
    prepared_dataset = datasets.load_from_disk("./waxal_processed_dataset")
else:
    print("Running preprocessing pipeline...")
    prepared_dataset = create_dataset_ultra_safe("google/WaxalNLP", lang_code, sr, max_len)
    print("Saving clean dataset to disk...")
    prepared_dataset.save_to_disk("./waxal_processed_dataset")

# Load evaluation metric
wer_metric = evaluate.load("wer")
gc.collect()
print("Data preparation complete.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.9 MB/s eta 0:00:00
Running preprocessing pipeline...
Loading sna dataset...


README.md:   0%|          | 0.00/31.5k [00:00<?, ?B/s]

data/ASR/sna/sna-train-00000.parquet: reconstructing file:   0%|          |  0.00B /  506MB            

data/ASR/sna/sna-train-00000.parquet: downloading bytes:           |  0.00B            

data/ASR/sna/sna-train-00001.parquet: reconstructing file:   0%|          |  0.00B /  506MB            

...

Map:   0%|          | 0/253 [00:00<?, ? examples/s]

Splitting dataset 80/20...
Saving clean dataset to disk...


Saving the dataset (0/1 shards):   0%|          | 0/202 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/51 [00:00<?, ? examples/s]

Data preparation complete.


In [6]:
import inspect
import transformers
import torch
import gc
import datasets
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from torch.utils.tensorboard import SummaryWriter

print("transformers version:", transformers.__version__)

# Free up RAM/VRAM before loading the heavy model
gc.collect()
torch.cuda.empty_cache()

model_id = "facebook/mms-1b-all"
lang_code = "sna"

print("Loading saved dataset from disk...")
dataset = datasets.load_from_disk("./waxal_processed_dataset")

print(f"Loading model and processor for {model_id}...")
processor = transformers.AutoProcessor.from_pretrained(model_id, target_lang=lang_code)

model = transformers.Wav2Vec2ForCTC.from_pretrained(
    model_id,
    target_lang=lang_code,
    ignore_mismatched_sizes=True,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

#  adapter-only fine-tuning
model.init_adapter_layers()
model.freeze_base_model()

adapter_weights = model._get_adapters()
for param in adapter_weights.values():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")

model.gradient_checkpointing_enable()
model.config.use_cache = False

# Custom Data Collator
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["audio"]["array"]} for feature in features]
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        label_features = [{"input_ids": self.processor.tokenizer(feature["transcription"]).input_ids} for feature in features]
        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

#  Build TrainingArguments in a version-safe way
base_kwargs = dict(
    output_dir="./mms-1b-waxal",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    eval_strategy="steps",
    num_train_epochs=1,
    max_steps=100,
    fp16=True,
    optim="adamw_bnb_8bit",
    save_steps=50,
    eval_steps=50,
    logging_steps=10,
    learning_rate=3e-4,
    warmup_steps=50,
    save_total_limit=1,
    report_to=["tensorboard"],
    dataloader_num_workers=0,
    remove_unused_columns=False,
    # group_by_length intentionally OFF: our dataset stores raw audio under
    # "audio", not "input_values", so the LengthGroupedSampler can't infer
    # lengths automatically. Turning it on again would require adding a
    # precomputed length column to the dataset (see note below).
)

sig_params = set(inspect.signature(transformers.TrainingArguments.__init__).parameters.keys())

if "eval_strategy" not in sig_params and "evaluation_strategy" in sig_params:
    base_kwargs["evaluation_strategy"] = base_kwargs.pop("eval_strategy")

final_kwargs = {k: v for k, v in base_kwargs.items() if k in sig_params}
dropped = set(base_kwargs) - set(final_kwargs)
if dropped:
    print(f"Dropped unsupported TrainingArguments kwargs for this version: {dropped}")

training_args = transformers.TrainingArguments(**final_kwargs)

writer = SummaryWriter(log_dir='./runs/mms')

trainer_sig = set(inspect.signature(transformers.Trainer.__init__).parameters.keys())
trainer_kwargs = dict(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = processor.feature_extractor
elif "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = processor.feature_extractor

trainer = transformers.Trainer(**trainer_kwargs)

print("Starting Training...")
trainer.train()

model.save_pretrained("./mms-1b-waxal-final", safe_serialization=True)
processor.save_pretrained("./mms-1b-waxal-final")
print("Training complete. Adapter + processor saved to ./mms-1b-waxal-final")

transformers version: 5.13.1
Loading saved dataset from disk...
Loading model and processor for facebook/mms-1b-all...


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154]) vs model:torch.Size([65])            
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154, 1280]) vs model:torch.Size([65, 1280])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable params: 2,234,433 / 964,731,841 (0.232%)
Starting Training...


Step,Training Loss,Validation Loss
50,59.688098,5.888193
100,13.487488,1.163860


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Adapter + processor saved to ./mms-1b-waxal-final
